# 03 - 误差分析（基于预测结果）

读取推理生成的 `predictions.csv`，统计 TP/FP/FN/TN，并导出漏报（FN）、误报（FP）清单。

**路径解析**：与 CLI 一致，优先 `runs/<run_name>/.latest_infer_session`，否则根目录 `predictions.csv`。

**实验切换**：改下方 `EXP_YAML` — F1 全量用 `exp_full_f1_ddp.yaml` / `exp_full_f1_sid.yaml`；召回/精度用 `exp_full_recall_*` / `exp_full_precision_*`。

**指标口径**：`metrics.json` **顶层 = 部署 KPI**（验证集阈值，与 infer 一致）；`oracle_on_test` 为测试集事后最优阈值，**仅作分析**。

**空 mcelog 归因**：preprocess 在 `meta_*.csv` 写入 `empty_mcelog_window`（特征窗内无 mcelog 事件）。本 Notebook 将 FN/FP 与该字段及 sid 交叉统计。

**输出**：`outputs/tables/error_*.csv`、`error_metrics_kpi.csv`、`error_by_empty_mcelog.csv`、`error_by_sid.csv`

**流水线关系**：需先 **infer**；完整流程见 **`04_run_pipelines.ipynb`**。

**Notebook 提示**：修改 Python 源码后请先重跑 setup 单元，或 **Kernel → Restart**，避免 `ImportError`（旧模块缓存）。


In [ ]:
from pathlib import Path

from dram2cnn.utils.notebook_bootstrap import notebook_setup

PROJECT_ROOT = notebook_setup(cwd=Path.cwd().resolve())

import os
import subprocess

import numpy as np

from dram2cnn.utils.config import load_experiment_config
from dram2cnn.utils.pipeline_helpers import splits_dir_for_exp
from dram2cnn.utils.preprocess_preflight import check_preprocess_raw_inputs, experiment_run_name
from dram2cnn.utils.run_layout import read_pointer_dir, resolve_checkpoint_dir

EXP_YAML = PROJECT_ROOT / "configs" / "experiments" / "exp_baseline.yaml"
OUT_T = PROJECT_ROOT / "outputs" / "tables"
OUT_T.mkdir(parents=True, exist_ok=True)


In [ ]:
import os
import subprocess
from pathlib import Path

import numpy as np

from dram2cnn.utils.experiment_guards import (
    assert_predictions_match_experiment,
    splits_match_processed,
)

try:
    _EXP_REL_STR = str(EXP_YAML.relative_to(PROJECT_ROOT))
except ValueError:
    _EXP_REL_STR = str(EXP_YAML.resolve())


def _run_shell_script(rel_script: str, arg: str) -> None:
    cmd = ["bash", str(PROJECT_ROOT / rel_script), arg]
    env = os.environ.copy()
    src_abs = str(PROJECT_ROOT / "src")
    pp = env.get("PYTHONPATH", "")
    env["PYTHONPATH"] = f"{src_abs}:{pp}" if pp else src_abs
    print("提示：执行", " ".join(cmd))
    subprocess.run(cmd, cwd=str(PROJECT_ROOT), env=env, check=True)


exp_path = EXP_YAML.resolve()

run_name = experiment_run_name(exp_path)
run_root = PROJECT_ROOT / "runs" / run_name
run_root.mkdir(parents=True, exist_ok=True)

infer_dir = read_pointer_dir(run_root, ".latest_infer_session", "latest_infer")
pred_path = None
if infer_dir is not None:
    cand = infer_dir / "predictions.csv"
    if cand.is_file():
        pred_path = cand
if pred_path is None and (run_root / "predictions.csv").is_file():
    infer_dir = run_root
    pred_path = infer_dir / "predictions.csv"

cfg = None
if pred_path is not None and pred_path.is_file():
    print("提示：已找到 predictions.csv，跳过预处理、训练与推理。")
    try:
        cfg = load_experiment_config(str(exp_path))
        assert_predictions_match_experiment(cfg, infer_dir, PROJECT_ROOT)
        print("提示：predictions_manifest 与当前 EXP_YAML 一致。")
    except FileNotFoundError as e:
        print(f"提示：无法加载实验配置做 manifest 校验（{e}）；若 predictions 来自其它实验，请改 EXP_YAML 或重跑 infer。")
    except ValueError as e:
        raise RuntimeError(f"predictions 与 EXP_YAML 不一致: {e}") from e
else:
    try:
        cfg = load_experiment_config(str(exp_path))
    except FileNotFoundError as e:
        raise FileNotFoundError(
            "加载实验配置失败（常见原因：data.yaml 中的 mcelog/tickets/index_features 等路径不存在）。\n"
            f"原始错误: {e}\n"
            "若仅需误差分析，请先将 predictions.csv 放到 runs/<run_name>/.latest_infer_session/ 下。"
        ) from e
    run_name = str(cfg["base"].get("run_name", run_name))
    run_root = PROJECT_ROOT / "runs" / run_name
    run_root.mkdir(parents=True, exist_ok=True)

    splits_dir = splits_dir_for_exp(exp_path, PROJECT_ROOT)

    def _split_x_shape_matches_model(sd: Path, model_cfg: dict) -> bool:
        xt = sd / "X_train.npy"
        if not xt.is_file():
            return False
        x = np.load(xt, mmap_mode="r")
        if x.ndim != 4:
            return False
        exp_c = int(model_cfg.get("in_channels", x.shape[1]))
        exp_h = int(model_cfg.get("grid_height", x.shape[2]))
        exp_w = int(model_cfg.get("grid_width", x.shape[3]))
        _n, c, h, w = x.shape
        return int(c) == exp_c and int(h) == exp_h and int(w) == exp_w

    def _splits_ready() -> bool:
        return splits_match_processed(cfg, PROJECT_ROOT)

    if not _splits_ready():
        check_preprocess_raw_inputs(exp_path)
        print(f"提示：{cfg['data']['splits_dir']} 不完整，自动运行预处理（耗时取决于原始数据规模）。")
        _run_shell_script("scripts/preprocess.sh", _EXP_REL_STR)
    elif not _split_x_shape_matches_model(splits_dir, cfg["model"]):
        x = np.load(splits_dir / "X_train.npy", mmap_mode="r")
        chw = tuple(int(x.shape[i]) for i in range(1, 4)) if x.ndim == 4 else x.shape
        mc = cfg["model"]
        exp = (int(mc.get("in_channels", 3)), int(mc.get("grid_height", 0)), int(mc.get("grid_width", 0)))
        print(
            f"提示：现有切分 X_train (C,H,W)={chw} 与 model.yaml 期望 {exp} 不一致，将重新运行 preprocess。"
        )
        _run_shell_script("scripts/preprocess.sh", _EXP_REL_STR)

    if not _splits_ready():
        raise RuntimeError(f"预处理后仍缺少 splits，请检查 data/raw 与配置：{splits_dir}")
    if not _split_x_shape_matches_model(splits_dir, cfg["model"]):
        raise RuntimeError(
            f"预处理后 X_train 形状仍与 model.yaml 不一致：{splits_dir}。"
            "请检查 grid_representation / model.in_channels|grid_height|grid_width 并重跑 preprocess。"
        )

    try:
        resolve_checkpoint_dir(PROJECT_ROOT, cfg["base"])
    except FileNotFoundError:
        print("提示：未找到 checkpoint，自动运行训练（可能非常久；与 scripts/train.sh 一致，含 DDP 逻辑）。")
        _run_shell_script("scripts/train.sh", _EXP_REL_STR)

    try:
        resolve_checkpoint_dir(PROJECT_ROOT, cfg["base"])
    except FileNotFoundError as exc:
        raise RuntimeError(f"训练结束后仍未解析到 checkpoint：{exc}") from exc

    infer_dir = read_pointer_dir(run_root, ".latest_infer_session", "latest_infer")
    pred_path = None
    if infer_dir is not None:
        cand = infer_dir / "predictions.csv"
        if cand.is_file():
            pred_path = cand
    if pred_path is None and (run_root / "predictions.csv").is_file():
        infer_dir = run_root
        pred_path = infer_dir / "predictions.csv"

    if pred_path is None or not pred_path.is_file():
        print("提示：未找到 predictions.csv，自动运行推理。")
        _run_shell_script("scripts/infer.sh", _EXP_REL_STR)
        infer_dir = read_pointer_dir(run_root, ".latest_infer_session", "latest_infer")
        pred_path = None
        if infer_dir is not None:
            cand = infer_dir / "predictions.csv"
            if cand.is_file():
                pred_path = cand
        if pred_path is None and (run_root / "predictions.csv").is_file():
            infer_dir = run_root
            pred_path = infer_dir / "predictions.csv"

    if pred_path is None or not pred_path.is_file():
        raise RuntimeError(f"推理后仍找不到 predictions.csv，请检查 {run_root}")

infer_dir = pred_path.parent
print("run_name:", run_name)
print("infer_dir:", infer_dir)
print("predictions:", pred_path)


In [ ]:
import pandas as pd

if "pred_path" not in globals() or pred_path is None:
    raise RuntimeError("请先自上而下运行上一单元（须成功解析 predictions）。")

df = pd.read_csv(pred_path)
need = {"y_true", "y_pred"}
if not need.issubset(df.columns):
    raise ValueError(f"predictions.csv 需含列 {need}，实际: {list(df.columns)}")

yt = df["y_true"].astype(int)
yp = df["y_pred"].astype(int)

tp = int(((yt == 1) & (yp == 1)).sum())
fp = int(((yt == 0) & (yp == 1)).sum())
fn = int(((yt == 1) & (yp == 0)).sum())
tn = int(((yt == 0) & (yp == 0)).sum())

summary = pd.DataFrame([
    {"name": "TP", "count": tp},
    {"name": "FP", "count": fp},
    {"name": "FN", "count": fn},
    {"name": "TN", "count": tn},
])
summary.to_csv(OUT_T / "error_confusion_counts.csv", index=False)
print(summary)

fn_df = df[(yt == 1) & (yp == 0)].copy()
fp_df = df[(yt == 0) & (yp == 1)].copy()
fn_df.to_csv(OUT_T / "error_FN_samples.csv", index=False)
fp_df.to_csv(OUT_T / "error_FP_samples.csv", index=False)
print("FN rows", len(fn_df), "FP rows", len(fp_df))
print("已写入 outputs/tables/error_*.csv")


In [ ]:
# 部署 KPI vs 测试集 oracle 阈值（读 evaluate 产出的 metrics.json）
import json

metrics_path = infer_dir / "metrics.json"
if not metrics_path.is_file():
    print("提示：未找到 metrics.json，可先运行: bash scripts/evaluate.sh", _EXP_REL_STR)
else:
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    deploy = metrics.get("deployment") or {}
    oracle = metrics.get("oracle_on_test") or {}
    rows = []
    for name, block in [("deployment（汇报用）", deploy), ("oracle_on_test（仅分析）", oracle)]:
        m = (block or {}).get("metrics") or {}
        rows.append({
            "scope": name,
            "threshold": (block or {}).get("threshold"),
            "f1": m.get("f1"),
            "precision": m.get("precision"),
            "recall": m.get("recall"),
            "roc_auc": m.get("roc_auc"),
        })
    kpi_df = pd.DataFrame(rows)
    kpi_df.to_csv(OUT_T / "error_metrics_kpi.csv", index=False)
    print(kpi_df.to_string(index=False))
    print("\n已写入 outputs/tables/error_metrics_kpi.csv")

In [ ]:
# FN/FP 与 empty_mcelog_window、sid 交叉分析
import numpy as np

exp_path = EXP_YAML.resolve()
cfg = globals().get("cfg") or load_experiment_config(str(exp_path))

splits_dir = splits_dir_for_exp(EXP_YAML, PROJECT_ROOT)
meta_test_path = splits_dir / "meta_test.csv"
if not meta_test_path.is_file():
    print("提示：无 meta_test.csv，跳过空窗/sid 交叉分析（请对该实验 preprocess）")
else:
    meta_test = pd.read_csv(meta_test_path)
    if "sample_index" in df.columns:
        work = df.merge(meta_test, left_on="sample_index", right_index=True, how="left", suffixes=("", "_meta"))
    else:
        work = df.copy()
        if len(work) == len(meta_test):
            for c in meta_test.columns:
                if c not in work.columns:
                    work[c] = meta_test[c].values
        else:
            print("警告：predictions 行数与 meta_test 不一致，无法安全 join")

    if "empty_mcelog_window" in work.columns:
        work["error_type"] = np.where(
            (work["y_true"] == 1) & (work["y_pred"] == 0), "FN",
            np.where((work["y_true"] == 0) & (work["y_pred"] == 1), "FP", "ok"),
        )
        err = work[work["error_type"].isin(["FN", "FP"])].copy()
        by_empty = (
            err.groupby(["error_type", "empty_mcelog_window"], dropna=False)
            .size()
            .reset_index(name="count")
        )
        by_empty.to_csv(OUT_T / "error_by_empty_mcelog.csv", index=False)
        print("=== FN/FP × empty_mcelog_window ===")
        print(by_empty.to_string(index=False))
        if "sid" in err.columns:
            by_sid = (
                err.groupby(["error_type", "sid"]).size().reset_index(name="count").sort_values("count", ascending=False)
            )
            by_sid.head(30).to_csv(OUT_T / "error_by_sid.csv", index=False)
            print("\n=== FN/FP 高发 sid（Top 15）===")
            print(by_sid.head(15).to_string(index=False))
    else:
        print("提示：meta 无 empty_mcelog_window 列；请对当前数据重跑 preprocess 以启用空窗归因。")